# Python + MySQL Implementation

Connect Python to the MySQL `employees` database, run basic queries, and process the large `salaries` table using a generator and an unbuffered cursor.

## 1. Install MySQL Connector

`mysql-connector-python` allows Python to connect to MySQL.

In [1]:
%pip install mysql-connector-python

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## 2. Import the Connector

Import the library used to create the MySQL connection.

In [2]:
import mysql.connector

## 3. Connect to MySQL

Connect to the local MySQL server and select the `employees` database.

In [3]:
connection = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password="amr123",
    database="employees"
)

print("Connected successfully!")

Connected successfully!


## 4. Create a Cursor

A cursor is used to execute SQL queries and retrieve results.

In [4]:
cursor = connection.cursor()

## 5. Execute a Basic Query

Run a simple query to check the number of employees.

In [5]:
cursor.execute("SELECT COUNT(*) FROM employees")
result = cursor.fetchone()
print("Employee count:", result[0])

Employee count: 300024


## 6. Fetch Sample Records

Use `LIMIT` to retrieve only a small number of rows.

In [6]:
cursor.execute("""
SELECT emp_no, first_name, last_name
FROM employees
LIMIT 10
""")

for row in cursor.fetchall():
    print(row)

(10001, 'Georgi', 'Facello')
(10002, 'Bezalel', 'Simmel')
(10003, 'Parto', 'Bamford')
(10004, 'Chirstian', 'Koblick')
(10005, 'Kyoichi', 'Maliniak')
(10006, 'Anneke', 'Preusig')
(10007, 'Tzvetan', 'Zielinski')
(10008, 'Saniya', 'Kalloufi')
(10009, 'Sumant', 'Peac')
(10010, 'Duangkaew', 'Piveteau')


## 7. Query the Large `salaries` Table

The `salaries` table contains about 28,44,047 records. Fetching all rows at once with `fetchall()` can use a large amount of memory.

In [7]:
cursor.execute("SELECT COUNT(*) FROM salaries")
salary_count = cursor.fetchone()[0]
print("Salary records:", salary_count)

Salary records: 2844047


## 8. Generator for Large Result Sets

A generator yields one row at a time instead of storing all rows in a list.

In [8]:
def fetch_salaries(cursor):
    cursor.execute("""
        SELECT emp_no, salary, from_date, to_date
        FROM salaries
    """)

    for row in cursor:
        yield row

## 9. Process Rows One at a Time

Only the current row is handled during iteration.

In [9]:
count = 0

for row in fetch_salaries(cursor):
    count += 1

print("Records processed:", count)

Records processed: 2844047


## 10. Memory-Efficient Processing

`buffered=False` creates an unbuffered cursor, so rows are read incrementally instead of buffering the complete result set in memory.

In [10]:
stream_cursor = connection.cursor(buffered=False)

stream_cursor.execute("""
    SELECT emp_no, salary, from_date, to_date
    FROM salaries
""")

count = 0
for row in stream_cursor:
    count += 1

print("Records processed:", count)

Records processed: 2844047


## 11. Generator + Unbuffered Cursor

This combines both ideas: the database rows are streamed and the generator processes them one at a time.

In [11]:
def stream_salaries(cursor):
    cursor.execute("""
        SELECT emp_no, salary, from_date, to_date
        FROM salaries
    """)

    for row in cursor:
        yield row

stream_cursor = connection.cursor(buffered=False)
count = 0

for row in stream_salaries(stream_cursor):
    # Process each row here
    count += 1

print("Records processed:", count)

Records processed: 2844047


## 12. Process Data Without Storing It

Example: calculate the total salary while reading rows one at a time.

In [12]:
stream_cursor = connection.cursor(buffered=False)
total_salary = 0
count = 0

for emp_no, salary, from_date, to_date in stream_salaries(stream_cursor):
    total_salary += salary
    count += 1

print("Records processed:", count)
print("Total salary:", total_salary)

Records processed: 2844047
Total salary: 181480757419


## 13. Cleanup

Close cursors and the database connection after processing.

In [13]:
cursor.close()
stream_cursor.close()
connection.close()

print("Connection closed.")

Connection closed.


## Key Points

- `mysql.connector` connects Python to MySQL.
- `cursor.execute()` runs SQL queries.
- `fetchone()` retrieves one row.
- `fetchall()` loads all returned rows into memory.
- A generator uses `yield` to produce rows one at a time.
- `buffered=False` allows incremental result processing.
- Generator + unbuffered cursor is suitable for large result sets.